# OasisSpaces — film a room, get a 3D model and a Gaussian splat

This notebook runs the whole OasisSpaces pipeline on Colab's GPU, one stage at a time:

1. **Reconstruct**: COLMAP finds where the phone was for every frame.
2. **Densify**: MoGe-2 measures depth in metres, GroundingDINO names the objects and SAM 2.1 cuts out their outlines, giving a dense, labelled point cloud.
3. **Shapes**: walls, floor, ceiling and furniture become an editable Blender room (`room.blend`).
4. **Splat**: OpenSplat trains a photorealistic Gaussian splat of the room.

After each stage a gate reports **PASS**, **WARN** or **STOP**. Look at the result before running the next stage, so a bad stage never costs the time of the ones after it.

**Before running:** menu *Runtime → Change runtime type → T4 GPU*.

**Capture tips:** walk slowly around the room in one loop and end where you started; don't turn on the spot. Keep the line where the floor meets the walls in view. Close curtains and avoid filming into mirrors or bright windows.

In [ ]:
!nvidia-smi -L || echo 'No GPU! Use Runtime -> Change runtime type -> T4 GPU'

## 1. Install the tools

Once per Colab session. The four cells together take roughly 15–25 minutes, most of it building OpenSplat.

In [ ]:
%%bash
# ffmpeg for video frames; COLMAP 4 (CUDA build from conda-forge) for camera solving.
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y ffmpeg libopencv-dev > /dev/null
if ! colmap -h 2>/dev/null | grep -q "COLMAP 4"; then
  cd /tmp
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
  export CONDA_OVERRIDE_CUDA=$(nvidia-smi | sed -n 's/.*CUDA Version: \([0-9.]*\).*/\1/p')
  # conda-forge's colmap 4.2.0 build omits its OpenImageIO dependency, so name it.
  ./bin/micromamba create -y -q -p /opt/colmap-env -c conda-forge "colmap=4.2.0=cuda*" "openimageio=3.1" \
    || { echo "No CUDA COLMAP for this driver; installing the CPU build (slower)";
         ./bin/micromamba create -y -q -p /opt/colmap-env -c conda-forge "colmap=4.2.0" "openimageio=3.1"; }
  ln -sf /opt/colmap-env/bin/colmap /usr/local/bin/colmap
fi
colmap -h 2>&1 | head -1

In [ ]:
%%bash
# MoGe-2 is pinned: newer releases change the model interface densify.py uses.
pip -q install 'numpy>=2.3' 'transformers>=5' anthropic
pip -q install --no-deps \
  "git+https://github.com/EasternJournalist/utils3d.git@3fab839f0be9931dac7c8488eb0e1600c236e183" \
  "git+https://github.com/microsoft/MoGe.git@925b8ed835a7a9cdb7578ba15c658a0afc969030"
python -c "import torch, transformers, moge.model.v2; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| transformers', transformers.__version__)"

In [ ]:
%%bash
# Blender builds the room model. Colab has no display, so it renders with Cycles on the CPU.
set -e
if [ ! -x /opt/blender/blender ]; then
  apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 > /dev/null
  curl -Ls https://download.blender.org/release/Blender5.2/blender-5.2.1-linux-x64.tar.xz | tar -xJ -C /opt
  mv /opt/blender-5.2.1-linux-x64 /opt/blender
fi
/opt/blender/blender --version | head -1

In [ ]:
%%bash
# OpenSplat (AGPL-3.0) trains the splat. Built against Colab's PyTorch for this GPU;
# pinned to the commit the pipeline was tested with.
set -e
if [ ! -x /content/OpenSplat/build/opensplat ]; then
  cd /content
  rm -rf OpenSplat
  git clone -q https://github.com/WebODM/OpenSplat.git
  cd OpenSplat && git checkout -q 2947938b7a2152b560489a963e47e961825dd32a
  export PATH=/usr/local/cuda/bin:$PATH
  TORCH_PREFIX=$(python -c "import torch; print(torch.utils.cmake_prefix_path)")
  ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
  mkdir -p build && cd build
  cmake -DCMAKE_PREFIX_PATH="$TORCH_PREFIX" -DGPU_RUNTIME=CUDA \
        -DCMAKE_CUDA_ARCHITECTURES="$ARCH" -DCMAKE_BUILD_TYPE=Release .. > cmake.log 2>&1 \
    || { tail -30 cmake.log; exit 1; }
  echo "Building OpenSplat (about 10 minutes)..."
  make -j"$(nproc)" opensplat > make.log 2>&1 || { tail -40 make.log; exit 1; }
fi
ls -la /content/OpenSplat/build/opensplat

## 2. Get the pipeline

In [ ]:
%cd /content
!rm -rf /content/OasisSpaces && git clone -q https://github.com/Oasis-spaces/OasisSpaces.git /content/OasisSpaces
%cd /content/OasisSpaces
!git log --oneline -1

## 3. Claude checks (optional)

With an Anthropic API key, Claude looks at frames and renders along the way. It picks a retry when the camera solve is weak, decides which wall and furniture candidates are real, and judges whether the built room is plausible. That is about six calls per room, with images, billed to your key.

To turn it on, add a secret named `ANTHROPIC_API_KEY` (key icon in the left bar) and allow this notebook to read it. Without it, every stage still runs and is judged by measurements alone.

In [ ]:
import os

USE_CLAUDE = False
try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
    if key:
        os.environ['ANTHROPIC_API_KEY'] = key
        USE_CLAUDE = True
except Exception as exc:  # no such secret, or the notebook was not given access
    print(f'No ANTHROPIC_API_KEY secret ({type(exc).__name__})')
print('Claude checks:', 'on' if USE_CLAUDE else 'off')

## 4. Upload your capture

Upload one walkthrough **video**, or a set of overlapping **photos**. Big videos upload faster through Google Drive: mount it with the folder icon on the left and set `source` to the file's Drive path instead.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
capture_dir = Path('/content/OasisSpaces/captures')
shutil.rmtree(capture_dir, ignore_errors=True)  # no stale uploads
capture_dir.mkdir(parents=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (capture_dir / name).write_bytes(data)
videos = [n for n in uploaded if Path(n).suffix.lower() in VIDEO_EXTENSIONS]
source = str(capture_dir / videos[-1]) if videos else str(capture_dir)
print('source =', source)

## 5. Run the stages

Run one cell at a time. Each ends with the gate:

- **PASS**: the output looks right; run the next stage.
- **WARN**: something is questionable (the reason is printed). Look at it before deciding to go on.
- **STOP**: the output is wrong; later stages would only waste time. The printed reason and the capture advice say what to change.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import torch
from IPython.display import Image, display

SPACE = 'colab-space'
space_dir = Path('/content/OasisSpaces/spaces') / SPACE
os.environ.update(
    BLENDER='/opt/blender/blender',
    OPENSPLAT='/content/OpenSplat/build/opensplat',
    OASIS_RENDER_ENGINE='CYCLES',
    LD_LIBRARY_PATH=os.path.join(os.path.dirname(torch.__file__), 'lib')
    + ':' + os.environ.get('LD_LIBRARY_PATH', ''),
)

def run_stage(stage):
    command = [sys.executable, 'pipeline/agent.py', source, '--name', SPACE, '--stage', stage]
    if not USE_CLAUDE:
        command.append('--no-claude')
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    report_path = space_dir / 'agent-report.json'
    gate = json.loads(report_path.read_text())['gates'].get(stage, {}) if report_path.exists() else {}
    status = gate.get('status', 'stop')
    print('\n' + '=' * 70)
    print(f"{stage}: {status.upper()} - {gate.get('why', 'the stage did not finish; see the log above')}")
    if status == 'stop':
        raise RuntimeError(f'{stage} stopped; fix this before running the next stage')

### Stage 1: reconstruct (cameras)

Extracts sharp frames (2 per second) and solves where the camera was for each. It passes when most frames land in one model. Frames that were placed badly in a quick pan are dropped automatically.

In [ ]:
run_stage('reconstruct')

### Stage 2: densify (metric, labelled cloud)

MoGe-2 predicts depth in metres for keyframes (one per ~7 frames), which also fixes the model's real-world scale. GroundingDINO names objects and SAM 2.1 cuts their outlines, so each point knows what it belongs to. Points on mirrors, windows and screens are dropped. It passes when keyframes agree on scale.

In [ ]:
run_stage('densify')
print(json.dumps({k: v for k, v in json.loads((space_dir / 'densify.json').read_text()).items()
                  if k in ('keyframes', 'colmap_units_per_metre', 'scale_spread', 'points', 'labels')}, indent=1))

### Stage 3: shapes (the room model)

Fits walls, floor and ceiling, turns labelled objects into furniture boxes, stands furniture on the floor, closes unseen sides with inferred walls, and builds `room.blend`. With Claude on, it also reviews the numbered candidates and judges the render. Check the renders below: walls meeting at the corners, furniture where it is in the video.

In [ ]:
run_stage('shapes')
for name in ('room-render.png', 'room-render-plan.png', 'plan-reviewed.png'):
    if (space_dir / name).exists():
        print(name)
        display(Image(filename=str(space_dir / name), width=720))

### Stage 4: splat (photorealistic view)

Trains a Gaussian splat for 10,000 steps, seeded from the dense cloud so plain walls start filled in. Writes `splat.ply` (full) and `splat.splat` (compact, for the web viewer).

In [ ]:
run_stage('splat')

## 6. Download the results

A zip with the splat, the Blender room, its renders, the room measurements (`shapes.json`) and the agent's report with capture advice. The dense cloud is large (hundreds of MB), so it is left out unless you set `INCLUDE_DENSE_CLOUD = True`.

In [ ]:
import zipfile
from google.colab import files

INCLUDE_DENSE_CLOUD = False
names = ['splat.splat', 'splat.ply', 'room.blend', 'room-render.png', 'room-render-plan.png',
         'plan-reviewed.png', 'shapes.json', 'densify.json', 'agent-report.json', 'cloud.ply']
if INCLUDE_DENSE_CLOUD:
    names.append('cloud-dense.ply')
archive = Path(f'/content/{SPACE}-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in names:
        if (space_dir / name).exists():
            zf.write(space_dir / name, arcname=name)
print(f'{archive} ({archive.stat().st_size / 1e6:.0f} MB)')
files.download(str(archive))

## 7. Look at it

- **Splat:** open the OasisSpaces splat viewer (`splat-viewer/index.html` in the repo, served with the website) and drag `splat.splat` onto the page. Drag with the mouse or use the arrow keys to move.
- **Room model:** open `room.blend` in Blender. Walls, floor and every piece of furniture are separate objects you can move, resize or restyle.
- **Measurements:** `shapes.json` holds the room in COLMAP units; divide by `colmap_units_per_metre` in `densify.json` for metres.
- **Capture advice:** `agent-report.json` lists what to change for a better next capture.